# Fine Structure of Two-Electron Atom He
## Atomic and Molecular Physics Laboratory - Experiment 06

This notebook contains the data reduction, wavelength determination, experimental grating constant extraction, spectrometer calibration regression, and artifact generation (`plots/` and `report/tables/`) for Experiment 06 within the scope of the laboratory manual.

### Objectives:
1. Determine experimental wavelengths $\lambda(d_{\text{manual}})$ at diffraction order $m = 1$ using the manual grating constant $d_{\text{manual}} = 1684\text{ nm}$ and $m \lambda = d \sin\theta_{\text{ave}}$.
2. Determine the experimental grating constant $d_i$ from each observed line and the mean $\bar{d}$ using the accepted manual reference wavelengths $\lambda_{\text{lit}}$.
3. Construct the $m = 1$ spectrometer calibration curve ($\lambda$ versus $\sin\theta_{\text{ave}}$) and compare all experimental determinations of $d$ and $\lambda$ against the accepted manual values.

In [1]:
import os
import io
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy import stats
import fitz
from PIL import Image

os.makedirs('../plots', exist_ok=True)
os.makedirs('../report/tables', exist_ok=True)

# extract Figure 1 (term diagram) and Figure 2 (spectrometer setup) from manual pdf
pdf_path = '../ATOMIC AND MOLECULAR PHYSICS LABORATORY-6th_.pdf'
if os.path.exists(pdf_path):
    doc = fitz.open(pdf_path)
    term_img_data = doc.extract_image(33)
    term_img = Image.open(io.BytesIO(term_img_data['image']))
    term_img.save('../plots/he_term_diagram.png')
    print(f"Extracted Figure 1 term diagram ({term_img.size}) to ../plots/he_term_diagram.png")

    setup_img_data = doc.extract_image(56)
    setup_img = Image.open(io.BytesIO(setup_img_data['image']))
    setup_img.save('../plots/he_spectrometer_setup.png')
    print(f"Extracted Figure 2 setup image ({setup_img.size}) to ../plots/he_spectrometer_setup.png")

Extracted Figure 1 term diagram ((716, 646)) to ../plots/he_term_diagram.png
Extracted Figure 2 setup image ((800, 354)) to ../plots/he_spectrometer_setup.png


### 1. Physical Constants and Apparatus Parameters
Fundamental constants and experimental parameters loaded from the raw dataset and manual.

In [2]:
# physical constants
h_planck_manual = 6.63e-34 # j*s, manual value
h_planck_si = 6.62607015e-34 # j*s, codata
c_light = 2.99792458e8 # m/s
e_charge = 1.602176634e-19 # c

# load raw measurements and apparatus metadata
df = pd.read_csv('../DATA/data.csv')
d_manual_nm = float(df['d_manual_nm'].iloc[0]) # nm, nominal manual grating constant
m_order = int(df['order_m'].iloc[0]) # diffraction order measured (m = 2 omitted)
theta_0_deg = float(df['theta_0_deg'].iloc[0]) # deg, zero-order goniometer angle
sigma_theta_deg = 0.1 # deg, goniometer vernier reading precision

print(f"Accepted manual grating constant d_manual: {d_manual_nm:.1f} nm")
print(f"Diffraction order m: {m_order}")
print(f"Recorded zero-order angle theta_0: {theta_0_deg:.1f} deg")
print(f"Goniometer reading uncertainty sigma_theta: +/- {sigma_theta_deg:.1f} deg")

Accepted manual grating constant d_manual: 1684.0 nm
Diffraction order m: 1
Recorded zero-order angle theta_0: 110.0 deg
Goniometer reading uncertainty sigma_theta: +/- 0.1 deg


### 2. Diffraction Angle Reduction, Wavelengths, and Experimental Grating Constant $d$
Computation of average diffraction angles $\theta_{\text{ave}}$, wavelengths using $d_{\text{manual}}$, line-by-line grating constants $d_i = m\lambda_{\text{lit},i}/\sin\theta_{\text{ave},i}$, recalculated wavelengths using $\bar{d}$, and transition photon energies.

In [3]:
# angular displacements relative to zero-order reading
df['diff_left_signed_deg'] = df['theta_left_deg'] - df['theta_0_deg']
df['diff_right_signed_deg'] = df['theta_right_deg'] - df['theta_0_deg']
df['diff_left_mag_deg'] = np.abs(df['diff_left_signed_deg'])
df['diff_right_mag_deg'] = np.abs(df['diff_right_signed_deg'])

# average diffraction angle and reading asymmetry
df['theta_ave_deg'] = 0.5 * (df['diff_left_mag_deg'] + df['diff_right_mag_deg'])
df['theta_mid_deg'] = 0.5 * (df['theta_left_deg'] + df['theta_right_deg'])
df['asym_deg'] = df['diff_right_mag_deg'] - df['diff_left_mag_deg']
df['sin_theta_ave'] = np.sin(np.deg2rad(df['theta_ave_deg']))

sigma_theta_ave_deg = sigma_theta_deg / np.sqrt(2.0)
sigma_theta_ave_rad = np.deg2rad(sigma_theta_ave_deg)
cos_theta_ave = np.cos(np.deg2rad(df['theta_ave_deg']))

# wavelengths computed using accepted manual grating constant d_manual
df['lambda_man_nm'] = (d_manual_nm / m_order) * df['sin_theta_ave']
df['sigma_lambda_man_nm'] = (d_manual_nm / m_order) * cos_theta_ave * sigma_theta_ave_rad
df['delta_lambda_man_nm'] = df['lambda_man_nm'] - df['lambda_lit_nm']
df['rel_err_man_pct'] = np.abs(df['delta_lambda_man_nm']) / df['lambda_lit_nm'] * 100.0

# experimental grating constant d_i determined from each reference line
df['d_calc_nm'] = (m_order * df['lambda_lit_nm']) / df['sin_theta_ave']
df['sigma_d_calc_nm'] = (m_order * df['lambda_lit_nm'] * cos_theta_ave / (df['sin_theta_ave'] ** 2)) * sigma_theta_ave_rad
df['d_calc_diff_pct'] = np.abs(df['d_calc_nm'] - d_manual_nm) / d_manual_nm * 100.0

d_bar_nm = df['d_calc_nm'].mean()
d_bar_std = df['d_calc_nm'].std(ddof=1)
d_bar_se = d_bar_std / np.sqrt(len(df))
d_bar_rel_diff_pct = np.abs(d_bar_nm - d_manual_nm) / d_manual_nm * 100.0

# wavelengths re-evaluated using experimental mean grating constant d_bar
df['lambda_dbar_nm'] = (d_bar_nm / m_order) * df['sin_theta_ave']
df['sigma_lambda_dbar_nm'] = (d_bar_nm / m_order) * cos_theta_ave * sigma_theta_ave_rad
df['sigma_lambda_dbar_comb_nm'] = np.sqrt(df['sigma_lambda_dbar_nm'] ** 2 + (df['sin_theta_ave'] * d_bar_se) ** 2)
df['delta_lambda_dbar_nm'] = df['lambda_dbar_nm'] - df['lambda_lit_nm']
df['rel_err_dbar_pct'] = np.abs(df['delta_lambda_dbar_nm']) / df['lambda_lit_nm'] * 100.0

# transition photon energies
hc_ev_nm = (h_planck_si * c_light / e_charge) * 1e9
df['energy_lit_ev'] = hc_ev_nm / df['lambda_lit_nm']
df['energy_man_ev'] = hc_ev_nm / df['lambda_man_nm']
df['energy_dbar_ev'] = hc_ev_nm / df['lambda_dbar_nm']

# summary metrics
theta_mid_mean = df['theta_mid_deg'].mean()
theta_mid_std = df['theta_mid_deg'].std(ddof=1)
asym_mean = df['asym_deg'].mean()
asym_std = df['asym_deg'].std(ddof=1)
mean_delta_man = df['delta_lambda_man_nm'].mean()
mean_rel_err_man = df['rel_err_man_pct'].mean()
min_rel_err_man = df['rel_err_man_pct'].min()
max_rel_err_man = df['rel_err_man_pct'].max()
mean_rel_err_dbar = df['rel_err_dbar_pct'].mean()
max_rel_err_dbar = df['rel_err_dbar_pct'].max()

print(f"Propagated uncertainty in theta_ave: +/- {sigma_theta_ave_deg:.4f} deg")
print(f"Mean midpoint angle: {theta_mid_mean:.2f} +/- {theta_mid_std:.2f} deg (vs theta_0 = {theta_0_deg:.1f} deg)")
print(f"Mean right-left deflection asymmetry: {asym_mean:.2f} +/- {asym_std:.2f} deg")
print(f"Manual d = {d_manual_nm:.1f} nm -> mean wavelength offset: +{mean_delta_man:.2f} nm, "
      f"mean rel. error: {mean_rel_err_man:.2f}% ({min_rel_err_man:.2f}% to {max_rel_err_man:.2f}%)")
print(f"Experimental mean d_bar: {d_bar_nm:.2f} +/- {d_bar_se:.2f} nm (s_d = {d_bar_std:.2f} nm), "
      f"diff vs manual d: {d_bar_rel_diff_pct:.2f}%")
print(f"Using d_bar -> mean wavelength rel. error: {mean_rel_err_dbar:.2f}% (max = {max_rel_err_dbar:.2f}%)")
for _, row in df.iterrows():
    print(f"  {row['color']:6s}: theta_ave={row['theta_ave_deg']:.2f} deg, sin={row['sin_theta_ave']:.4f}, "
          f"lam_man={row['lambda_man_nm']:.1f} +/- {row['sigma_lambda_man_nm']:.1f} nm (err={row['rel_err_man_pct']:.2f}%), "
          f"d_i={row['d_calc_nm']:.1f} +/- {row['sigma_d_calc_nm']:.1f} nm (diff={row['d_calc_diff_pct']:.2f}%), "
          f"lam_dbar={row['lambda_dbar_nm']:.1f} +/- {row['sigma_lambda_dbar_comb_nm']:.1f} nm (err={row['rel_err_dbar_pct']:.2f}%), "
          f"lam_lit={row['lambda_lit_nm']:.1f} nm")
display(df[['color', 'theta_ave_deg', 'sin_theta_ave', 'lambda_lit_nm', 'lambda_man_nm', 'rel_err_man_pct', 'd_calc_nm', 'lambda_dbar_nm', 'rel_err_dbar_pct']])

Propagated uncertainty in theta_ave: +/- 0.0707 deg
Mean midpoint angle: 113.26 +/- 0.54 deg (vs theta_0 = 110.0 deg)
Mean right-left deflection asymmetry: 6.52 +/- 1.07 deg
Manual d = 1684.0 nm -> mean wavelength offset: +69.24 nm, mean rel. error: 13.66% (10.35% to 15.72%)
Experimental mean d_bar: 1482.01 +/- 12.88 nm (s_d = 28.80 nm), diff vs manual d: 11.99%
Using d_bar -> mean wavelength rel. error: 1.44% (max = 2.89%)
  Red   : theta_ave=25.95 deg, sin=0.4376, lam_man=736.9 +/- 1.9 nm (err=10.35%), d_i=1526.1 +/- 3.9 nm (diff=9.38%), lam_dbar=648.5 +/- 5.9 nm (err=2.89%), lam_lit=667.8 nm
  Yellow: theta_ave=23.20 deg, sin=0.3939, lam_man=663.4 +/- 1.9 nm (err=12.90%), d_i=1491.6 +/- 4.3 nm (diff=11.43%), lam_dbar=583.8 +/- 5.3 nm (err=0.64%), lam_lit=587.6 nm
  Green : theta_ave=20.25 deg, sin=0.3461, lam_man=582.9 +/- 1.9 nm (err=15.46%), d_i=1458.5 +/- 4.9 nm (diff=13.39%), lam_dbar=512.9 +/- 4.8 nm (err=1.61%), lam_lit=504.8 nm
  Blue  : theta_ave=17.60 deg, sin=0.3024, lam_m

,color,theta_ave_deg,sin_theta_ave,lambda_lit_nm,lambda_man_nm,rel_err_man_pct,d_calc_nm,lambda_dbar_nm,rel_err_dbar_pct
0,Red,25.95,0.437587,667.8,736.895892,10.346794,1526.097801,648.508816,2.888767
1,Yellow,23.20,0.393942,587.6,663.398176,12.899621,1491.590475,583.826793,0.642139
2,Green,20.25,0.346117,504.8,582.861124,15.463773,1458.466116,512.949769,1.614455
3,Blue,17.60,0.302370,447.1,509.190896,13.887474,1478.652517,448.115926,0.227226
4,Purple,15.50,0.267238,388.9,450.029425,15.718546,1455.255064,396.050586,1.838669


### 3. Spectrometer Calibration Curve ($m = 1$)
Linear regression of reference wavelength $\lambda_{\text{lit}}$ versus $\sin\theta_{\text{ave}}$ to determine the calibrated grating constant $d_{\text{cal}}$ (unconstrained OLS) and $d_{\text{eff}}$ (zero-intercept proportionality), and compare against $d_{\text{manual}} = 1684.0\text{ nm}$.

In [4]:
x_sin = df['sin_theta_ave'].values
y_lam = df['lambda_lit_nm'].values

# unconstrained ols calibration fit: lambda_lit = d_cal * sin(theta_ave) + intercept
reg = stats.linregress(x_sin, y_lam)
d_cal_nm = reg.slope
d_cal_se = reg.stderr
intercept_nm = reg.intercept
intercept_se = reg.intercept_stderr
r2_cal = reg.rvalue ** 2
p_cal = reg.pvalue
rms_cal_nm = np.sqrt(np.mean((y_lam - (d_cal_nm * x_sin + intercept_nm)) ** 2))

# inverse regression: sin(theta_ave) = m_inv * lambda_lit + b_inv
reg_inv = stats.linregress(y_lam, x_sin)
slope_inv = reg_inv.slope
slope_inv_se = reg_inv.stderr
intercept_inv = reg_inv.intercept
intercept_inv_se = reg_inv.intercept_stderr

# zero-intercept proportional fit: lambda_lit = d_eff * sin(theta_ave)
d_eff_nm = np.sum(x_sin * y_lam) / np.sum(x_sin ** 2)
resid_zero = y_lam - d_eff_nm * x_sin
d_eff_se = np.sqrt(np.sum(resid_zero ** 2) / (len(x_sin) - 1) / np.sum(x_sin ** 2))
ss_tot = np.sum((y_lam - np.mean(y_lam)) ** 2)
r2_zero = 1.0 - np.sum(resid_zero ** 2) / ss_tot
rms_zero_nm = np.sqrt(np.mean(resid_zero ** 2))

d_cal_diff_pct = np.abs(d_cal_nm - d_manual_nm) / d_manual_nm * 100.0
d_eff_diff_pct = np.abs(d_eff_nm - d_manual_nm) / d_manual_nm * 100.0

print(f"Unconstrained OLS: lambda = ({d_cal_nm:.2f} +/- {d_cal_se:.2f}) * sin(theta_ave) + ({intercept_nm:.2f} +/- {intercept_se:.2f}) nm")
print(f"  R^2 = {r2_cal:.6f}, p-value = {p_cal:.3e}, RMS residual = {rms_cal_nm:.2f} nm")
print(f"  d_cal diff vs manual d ({d_manual_nm:.1f} nm): {d_cal_diff_pct:.2f}%")
print(f"Inverse OLS: sin(theta_ave) = ({slope_inv:.6e} +/- {slope_inv_se:.2e}) * lambda + ({intercept_inv:.4f} +/- {intercept_inv_se:.4f})")
print(f"Zero-intercept fit: lambda = ({d_eff_nm:.2f} +/- {d_eff_se:.2f}) * sin(theta_ave) nm")
print(f"  R^2 (zero-intercept) = {r2_zero:.6f}, RMS residual = {rms_zero_nm:.2f} nm, diff vs manual d: {d_eff_diff_pct:.2f}%")

Unconstrained OLS: lambda = (1617.11 +/- 52.33) * sin(theta_ave) + (-45.86 +/- 18.57) nm
  R^2 = 0.996868, p-value = 7.446e-05, RMS residual = 5.55 nm
  d_cal diff vs manual d (1684.0 nm): 3.97%
Inverse OLS: sin(theta_ave) = (6.164489e-04 +/- 1.99e-05) * lambda + (0.0294 +/- 0.0105)
Zero-intercept fit: lambda = (1489.78 +/- 13.62) * sin(theta_ave) nm
  R^2 (zero-intercept) = 0.990498, RMS residual = 9.66 nm, diff vs manual d: 11.53%


### 4. Publication-Grade Figure Generation
Exporting analysis graphics directly to `../plots/`.

In [5]:
# Figure 1: Spectrometer calibration curve at m = 1
plt.figure(figsize=(7.2, 5.0), dpi=300)

color_hex = {
    'Red': '#d62728',
    'Yellow': '#d4a017',
    'Green': '#2ca02c',
    'Blue': '#1f77b4',
    'Purple': '#7b2cbf'
}

x_grid = np.linspace(0.24, 0.47, 200)

# manual grating equation line and calibration fits
plt.plot(x_grid, d_manual_nm * x_grid, color='#444444', linestyle='-', lw=1.8,
         label=f'Manual Grating Eq. ($d_{{manual}} = {d_manual_nm:.0f}$ nm)', zorder=2)
plt.plot(x_grid, d_cal_nm * x_grid + intercept_nm, color='#000000', linestyle='--', lw=2.0,
         label=f'OLS Calibration ($d_{{cal}} = {d_cal_nm:.1f}$ nm, $R^2 = {r2_cal:.4f}$)', zorder=3)
plt.plot(x_grid, d_eff_nm * x_grid, color='#7f7f7f', linestyle=':', lw=1.7,
         label=f'Zero-Intercept Fit ($d_{{eff}} = {d_eff_nm:.1f}$ nm)', zorder=2)

label_offsets = {
    'Red': (0, -28),
    'Yellow': (0, -26),
    'Green': (0, -25),
    'Blue': (0, -25),
    'Purple': (10, -12)
}

for _, row in df.iterrows():
    c = color_hex[row['color']]
    sys_tag = 'S' if row['multiplicity'] == 'Singlet' else 'T'
    plt.errorbar(row['sin_theta_ave'], row['lambda_man_nm'], yerr=row['sigma_lambda_man_nm'],
                 fmt='o', color=c, markeredgecolor='black', markersize=7.5, capsize=3, zorder=5)
    plt.scatter(row['sin_theta_ave'], row['lambda_lit_nm'], marker='s', s=55,
                facecolors='white', edgecolors=c, linewidths=1.8, zorder=5)
    plt.plot([row['sin_theta_ave'], row['sin_theta_ave']],
             [row['lambda_lit_nm'], row['lambda_man_nm']],
             color=c, linestyle=':', lw=1.0, alpha=0.7, zorder=4)
    dx, dy = label_offsets[row['color']]
    ha_align = 'left' if row['color'] == 'Purple' else 'center'
    plt.annotate(f"{row['color']} ({sys_tag})",
                 xy=(row['sin_theta_ave'], row['lambda_lit_nm']),
                 xytext=(dx, dy), textcoords='offset points', ha=ha_align,
                 fontsize=8.5, fontweight='bold', color=c)

plt.scatter([], [], marker='o', s=55, color='#333333', edgecolor='black', label=r'Experimental $\lambda(d_{\mathrm{manual}})$')
plt.scatter([], [], marker='s', s=55, facecolor='white', edgecolor='#333333', linewidth=1.8, label=r'Manual Reference $\lambda_{\mathrm{lit}}$ (S=Singlet, T=Triplet)')

eq_box = (f"OLS Fit: $\\lambda = ({d_cal_nm:.1f} \\pm {d_cal_se:.1f})\\sin\\theta_{{\\mathrm{{ave}}}} + ({intercept_nm:.1f} \\pm {intercept_se:.1f})$ nm\n"
          f"$R^2 = {r2_cal:.4f}$, RMS residual $= {rms_cal_nm:.2f}$ nm\n"
          f"Zero-Intercept: $d_{{\\mathrm{{eff}}}} = {d_eff_nm:.1f} \\pm {d_eff_se:.1f}$ nm")
plt.text(0.03, 0.96, eq_box, transform=plt.gca().transAxes, va='top', fontsize=8.3,
         bbox=dict(boxstyle='round,pad=0.35', facecolor='#f8f9fa', edgecolor='#bbbbbb', alpha=0.95))

plt.title('Spectrometer Calibration Curve for He I Spectrum ($m = 1$)', fontsize=11.5, fontweight='bold', pad=10)
plt.xlabel(r'$\sin\theta_{\mathrm{ave}}$', fontsize=10.5)
plt.ylabel(r'Wavelength $\lambda$ (nm)', fontsize=10.5)
plt.xlim(0.245, 0.465)
plt.ylim(350, 785)
plt.grid(True, linestyle=':', alpha=0.6)
plt.legend(loc='lower right', fontsize=7.8, framealpha=0.95)
plt.tight_layout()
plt.savefig('../plots/calibration_curve_m1.png')
plt.close()
print("Saved ../plots/calibration_curve_m1.png")

Saved ../plots/calibration_curve_m1.png


In [6]:
# Figure 2: Experimental grating constant d comparison against accepted manual value
plt.figure(figsize=(7.2, 4.8), dpi=300)

x_pos = np.arange(len(df))
for i, row in df.iterrows():
    c = color_hex[row['color']]
    plt.errorbar(i, row['d_calc_nm'], yerr=row['sigma_d_calc_nm'], fmt='o',
                 color=c, markeredgecolor='black', ecolor=c, elinewidth=1.8,
                 capsize=5, markersize=8.5, zorder=5)
    plt.text(i, row['d_calc_nm'] + row['sigma_d_calc_nm'] + 7.0, f"{row['d_calc_nm']:.1f}",
             ha='center', va='bottom', fontsize=8.5, fontweight='bold', color='#222222')

# reference and summary horizontal lines
plt.axhline(d_manual_nm, color='#d62728', linestyle='--', lw=2.0,
            label=f'Manual Accepted Value ($d_{{manual}} = {d_manual_nm:.1f}$ nm)', zorder=3)
plt.axhline(d_bar_nm, color='#1f77b4', linestyle='-', lw=1.9,
            label=f'Mean Experimental $\\bar{{d}} = {d_bar_nm:.1f} \\pm {d_bar_se:.1f}$ nm ($- {d_bar_rel_diff_pct:.2f}$%)', zorder=4)
plt.axhspan(d_bar_nm - d_bar_se, d_bar_nm + d_bar_se, color='#1f77b4', alpha=0.14,
            label=r'$\pm 1\sigma_{\bar{d}}$ Standard Error Band', zorder=1)
plt.axhline(d_cal_nm, color='#2ca02c', linestyle='-.', lw=1.8,
            label=f'OLS Calibration Slope ($d_{{cal}} = {d_cal_nm:.1f} \\pm {d_cal_se:.1f}$ nm)', zorder=3)

plt.errorbar([], [], yerr=[], fmt='o', color='#333333', markeredgecolor='black',
             capsize=4, label=r'Individual Line $d_i = \lambda_{\mathrm{lit},i} / \sin\theta_{\mathrm{ave},i}$')

plt.xticks(x_pos, df['color'], fontsize=10)
plt.xlim(-0.5, len(df) - 0.5)
plt.ylim(1415, 1735)
plt.ylabel('Grating Constant $d$ (nm)', fontsize=10.5)
plt.xlabel('Observed Helium Spectral Line', fontsize=10.5)
plt.title('Experimental Grating Constant $d$ Compared with Accepted Manual Value', fontsize=11.0, fontweight='bold', pad=10)
plt.grid(True, linestyle=':', alpha=0.6)
plt.legend(loc='center right', bbox_to_anchor=(0.99, 0.56), fontsize=8.0, framealpha=0.95)
plt.tight_layout()
plt.savefig('../plots/grating_constant_comparison.png')
plt.close()
print("Saved ../plots/grating_constant_comparison.png")

Saved ../plots/grating_constant_comparison.png


In [7]:
# Figure 3: Wavelength comparison (d_manual vs d_bar vs literature) and spectrum strip
fig = plt.figure(figsize=(7.6, 6.4), dpi=300)
gs = gridspec.GridSpec(2, 1, height_ratios=[2.1, 1.1], hspace=0.40)

ax1 = fig.add_subplot(gs[0])
idx = np.arange(len(df))
bar_w = 0.25
bar_colors = [color_hex[c] for c in df['color']]

ax1.bar(idx - bar_w, df['lambda_man_nm'], width=bar_w, yerr=df['sigma_lambda_man_nm'],
        color=bar_colors, edgecolor='black', linewidth=0.9, capsize=2.5,
        label=r'Using Manual $d_{\mathrm{manual}} = 1684$ nm')
ax1.bar(idx, df['lambda_dbar_nm'], width=bar_w, yerr=df['sigma_lambda_dbar_comb_nm'],
        color=bar_colors, alpha=0.45, edgecolor='black', linewidth=0.9, capsize=2.5,
        label=f'Using Mean $\\bar{{d}} = {d_bar_nm:.1f}$ nm')
ax1.bar(idx + bar_w, df['lambda_lit_nm'], width=bar_w,
        color='white', edgecolor=bar_colors, linewidth=1.7, hatch='///',
        label=r'Accepted Manual Reference $\lambda_{\mathrm{lit}}$')

trans_labels = [
    r'$3\,^1D \to 2\,^1P$',
    r'$3\,^3D \to 2\,^3P$',
    r'$4\,^1S \to 2\,^1P$',
    r'$4\,^3D \to 2\,^3P$',
    r'$3\,^3P \to 2\,^3S$'
]

for i, row in df.iterrows():
    ax1.text(i - bar_w, row['lambda_man_nm'] + 10.0, f"+{row['rel_err_man_pct']:.1f}%",
             ha='center', va='bottom', fontsize=7.6, fontweight='bold', color='#222222')
    y_mid_top = max(row['lambda_dbar_nm'], row['lambda_lit_nm']) + 8.0
    ax1.text(i + 0.5 * bar_w, y_mid_top, f"{row['rel_err_dbar_pct']:.1f}%",
             ha='center', va='bottom', fontsize=7.4, fontweight='bold', color='#1b4f72')

ax1.set_xticks(idx)
ax1.set_xticklabels([f"{c}\n({t})" for c, t in zip(df['color'], trans_labels)], fontsize=8.8)
ax1.set_ylabel(r'Wavelength $\lambda$ (nm)', fontsize=10.5)
ax1.set_title(r'Comparison of Wavelengths from $d_{\mathrm{manual}}$ and $\bar{d}$ Against Reference Values', fontsize=11.0, fontweight='bold', pad=10)
ax1.set_ylim(0, 845)
ax1.grid(axis='y', linestyle=':', alpha=0.6)
ax1.legend(loc='upper right', fontsize=8.2, framealpha=0.95)

# bottom panel: dark-field spectrum strip showing alignment after d calibration
ax2 = fig.add_subplot(gs[1])
ax2.set_facecolor('#111111')
ax2.axhline(0.66, color='#555555', lw=0.8)
ax2.axhline(0.33, color='#555555', lw=0.8)

for _, row in df.iterrows():
    c = color_hex[row['color']]
    ax2.plot([row['lambda_lit_nm'], row['lambda_lit_nm']], [0.69, 0.96], color=c, lw=2.5)
    ax2.plot([row['lambda_dbar_nm'], row['lambda_dbar_nm']], [0.36, 0.63], color=c, lw=2.5, linestyle='-.')
    ax2.plot([row['lambda_man_nm'], row['lambda_man_nm']], [0.04, 0.30], color=c, lw=2.5, linestyle='--')

ax2.text(336, 0.80, 'Manual Lit.', color='white', fontsize=7.8, fontweight='bold')
ax2.text(336, 0.47, r'Calibrated $\bar{d}$', color='white', fontsize=7.8, fontweight='bold')
ax2.text(336, 0.14, r'Nominal $d_{\mathrm{man}}$', color='white', fontsize=7.8, fontweight='bold')
ax2.set_xlim(330, 770)
ax2.set_ylim(0, 1)
ax2.set_yticks([])
ax2.set_xlabel(r'Wavelength $\lambda$ (nm)', fontsize=10.5)
ax2.set_title(r'Visible Helium Spectrum Alignment: Reference vs. Calibrated $\bar{d}$ vs. Manual $d$', fontsize=10.0, fontweight='bold', pad=6)

fig.subplots_adjust(left=0.10, right=0.97, top=0.92, bottom=0.10)
plt.savefig('../plots/wavelength_comparison.png')
plt.close()
print("Saved ../plots/wavelength_comparison.png")

Saved ../plots/wavelength_comparison.png


### 5. Table Artifact Exports for LaTeX Integration
Exporting formatted tables into `../report/tables/`.

In [8]:
# Table 1: manual table 3 structure for m = 1
table1_export = pd.DataFrame({
    'Color': df['color'],
    'theta_0': [f"{x:.1f}" for x in df['theta_0_deg']],
    'theta_left': [f"{x:.1f}" for x in df['theta_left_deg']],
    'diff_left': [f"{x:.1f}" for x in df['diff_left_signed_deg']],
    'theta_right': [f"{x:.1f}" for x in df['theta_right_deg']],
    'diff_right': [f"{x:.1f}" for x in df['diff_right_signed_deg']],
    'theta_ave': [f"{x:.2f}" for x in df['theta_ave_deg']],
    'sin_theta_ave': [f"{x:.4f}" for x in df['sin_theta_ave']],
    'lambda_exp': [f"{x:.1f}" for x in df['lambda_man_nm']]
})
table1_export.to_csv('../report/tables/table1_m1_results.csv', index=False)
print("Table 1 exported:")
display(table1_export)

# Table 2: comparison of manual-d wavelengths, calculated d_i, and d_bar wavelengths vs literature
latex_transitions = [
    '$3\\;^{1}D \\rightarrow 2\\;^{1}P$',
    '$3\\;^{3}D \\rightarrow 2\\;^{3}P$',
    '$4\\;^{1}S \\rightarrow 2\\;^{1}P$',
    '$4\\;^{3}D \\rightarrow 2\\;^{3}P$',
    '$3\\;^{3}P \\rightarrow 2\\;^{3}S$'
]
table2_export = pd.DataFrame({
    'Color': df['color'],
    'Transition': latex_transitions,
    'System': df['multiplicity'],
    'lambda_lit': [f"{x:.1f}" for x in df['lambda_lit_nm']],
    'lambda_man': [f"{v:.1f} $\\pm$ {s:.1f}" for v, s in zip(df['lambda_man_nm'], df['sigma_lambda_man_nm'])],
    'err_man_pct': [f"{x:.2f}" for x in df['rel_err_man_pct']],
    'd_calc': [f"{v:.1f} $\\pm$ {s:.1f}" for v, s in zip(df['d_calc_nm'], df['sigma_d_calc_nm'])],
    'lambda_dbar': [f"{v:.1f} $\\pm$ {s:.1f}" for v, s in zip(df['lambda_dbar_nm'], df['sigma_lambda_dbar_comb_nm'])],
    'err_dbar_pct': [f"{x:.2f}" for x in df['rel_err_dbar_pct']]
})
table2_export.to_csv('../report/tables/table2_literature_comparison.csv', index=False)
print("\nTable 2 exported:")
display(table2_export)

# Table 3: summary comparison of experimental d determinations against manual value
table3_export = pd.DataFrame({
    'Parameter': [
        'Accepted Manual Constant ($d_{\\text{manual}}$)',
        'Mean Experimental Constant ($\\bar{d}$)',
        'Sample Std. Dev. of $d_i$ ($s_d$)',
        'OLS Calibration Slope ($d_{\\text{cal}}$)',
        'OLS Calibration Intercept ($b$)',
        'OLS Coefficient of Determination ($R^2$)',
        'Zero-Intercept Slope ($d_{\\text{eff}}$)',
        'Mean $\\lambda$ Rel. Error (Manual $d$)',
        'Mean $\\lambda$ Rel. Error (Mean $\\bar{d}$)'
    ],
    'Value': [
        f"{d_manual_nm:.1f}",
        f"{d_bar_nm:.1f} $\\pm$ {d_bar_se:.1f}",
        f"{d_bar_std:.1f}",
        f"{d_cal_nm:.1f} $\\pm$ {d_cal_se:.1f}",
        f"{intercept_nm:.1f} $\\pm$ {intercept_se:.1f}",
        f"{r2_cal:.4f}",
        f"{d_eff_nm:.1f} $\\pm$ {d_eff_se:.1f}",
        f"{mean_rel_err_man:.2f}",
        f"{mean_rel_err_dbar:.2f}"
    ],
    'Unit': [
        'nm',
        'nm',
        'nm',
        'nm',
        'nm',
        '---',
        'nm',
        '\%',
        '\%'
    ],
    'Diff_vs_Manual': [
        '0.00\%',
        f"{d_bar_rel_diff_pct:.2f}\%",
        '---',
        f"{d_cal_diff_pct:.2f}\%",
        '---',
        '---',
        f"{d_eff_diff_pct:.2f}\%",
        '---',
        '---'
    ]
})
table3_export.to_csv('../report/tables/table3_calibration_summary.csv', index=False)
print("\nTable 3 exported:")
display(table3_export)

Table 1 exported:


,Color,theta_0,theta_left,diff_left,theta_right,diff_right,theta_ave,sin_theta_ave,lambda_exp
0,Red,110.0,88.0,-22.0,139.9,29.9,25.95,0.4376,736.9
1,Yellow,110.0,90.4,-19.6,136.8,26.8,23.20,0.3939,663.4
2,Green,110.0,92.3,-17.7,132.8,22.8,20.25,0.3461,582.9
3,Blue,110.0,95.5,-14.5,130.7,20.7,17.60,0.3024,509.2
4,Purple,110.0,97.6,-12.4,128.6,18.6,15.50,0.2672,450.0



Table 2 exported:


,Color,Transition,System,lambda_lit,lambda_man,err_man_pct,d_calc,lambda_dbar,err_dbar_pct
0,Red,$3\;^{1}D \rightarrow 2\;^{1}P$,Singlet,667.8,736.9 $\pm$ 1.9,10.35,1526.1 $\pm$ 3.9,648.5 $\pm$ 5.9,2.89
1,Yellow,$3\;^{3}D \rightarrow 2\;^{3}P$,Triplet,587.6,663.4 $\pm$ 1.9,12.90,1491.6 $\pm$ 4.3,583.8 $\pm$ 5.3,0.64
2,Green,$4\;^{1}S \rightarrow 2\;^{1}P$,Singlet,504.8,582.9 $\pm$ 1.9,15.46,1458.5 $\pm$ 4.9,512.9 $\pm$ 4.8,1.61
3,Blue,$4\;^{3}D \rightarrow 2\;^{3}P$,Triplet,447.1,509.2 $\pm$ 2.0,13.89,1478.7 $\pm$ 5.8,448.1 $\pm$ 4.3,0.23
4,Purple,$3\;^{3}P \rightarrow 2\;^{3}S$,Triplet,388.9,450.0 $\pm$ 2.0,15.72,1455.3 $\pm$ 6.5,396.1 $\pm$ 3.9,1.84



Table 3 exported:


,Parameter,Value,Unit,Diff_vs_Manual
0,Accepted Manual Constant ($d_{\text{manual}}$),1684.0,nm,0.00\%
1,Mean Experimental Constant ($\bar{d}$),1482.0 $\pm$ 12.9,nm,11.99\%
2,Sample Std. Dev. of $d_i$ ($s_d$),28.8,nm,---
3,OLS Calibration Slope ($d_{\text{cal}}$),1617.1 $\pm$ 52.3,nm,3.97\%
4,OLS Calibration Intercept ($b$),-45.9 $\pm$ 18.6,nm,---
5,OLS Coefficient of Determination ($R^2$),0.9969,---,---
6,Zero-Intercept Slope ($d_{\text{eff}}$),1489.8 $\pm$ 13.6,nm,11.53\%
7,Mean $\lambda$ Rel. Error (Manual $d$),13.66,\%,---
8,Mean $\lambda$ Rel. Error (Mean $\bar{d}$),1.44,\%,---
